## READ ME FIRST

This notebook produces a weekly release from the `develop` branch of Cytoscape.

The weekly release consists of Install4j generated installers that can be put on a public server to be downloaded by testers. This script will NOT push any code changes or resources to GitHub or Nexus or update any additional resources like the manual or API docs. 

This notebook is configured for the build server on which it is run. Notes have been made where local environment variables have been set.

Before starting this process, it is a good idea to Restart and Clear Output for this notebook's Kernel.

### Required environment

This notebook reads its configuration from the environment. Jupyter inherits the
environment of the shell that launches it, so export these **before** starting it —
setting them from inside a cell is too late.

```bash
export JAVA_HOME=/opt/jdk-17
export MAVEN_HOME=/opt/maven
export STARTING_BRANCH=develop
# optional
export PUBLISH_ROOT=/var/www/html/cytoscape-builds
export ADMIN_SCRIPTS_DIR=/home/cybuilder/cytoscape-admin-scripts

jupyter lab --no-browser --port 8888
```

### 1. Set up Build Environment

This sets up the build environment, including Java and Maven versions, and the root directory of the build.


In [1]:
from subprocess import Popen, PIPE, STDOUT
import os
import shutil
import datetime
import xml.etree.ElementTree as ET

Set the notebook directory. Note that this is system dependent.

In [2]:
# Where this admin-scripts checkout lives.  Override with ADMIN_SCRIPTS_DIR to run
# the notebook from a test checkout instead of the one cron uses.
NOTEBOOK_DIR = os.environ.get('ADMIN_SCRIPTS_DIR', '/home/cybuilder/cytoscape-admin-scripts')
assert os.path.isdir(NOTEBOOK_DIR), f'not a directory: {NOTEBOOK_DIR}'
print('NOTEBOOK_DIR =', NOTEBOOK_DIR)

/home/cybuilder/cytoscape-admin-scripts


Set the Java environment.

`JAVA_HOME` is read from the environment rather than pinned here, so this notebook
hard-codes no JDK. Export it in the shell that launches jupyter — see READ ME FIRST.
Cytoscape 3.11 compiles with `--release 17`, so it must point at a JDK 17 or newer.

In [3]:
# JAVA_HOME comes from the environment so this notebook pins no toolchain.
# Export it in the shell that launches jupyter, e.g.
#   export JAVA_HOME=/opt/jdk-17
JAVA_HOME = os.environ.get('JAVA_HOME', '').strip()
assert JAVA_HOME, 'JAVA_HOME is not set - export it before launching jupyter'
# javac, not java: maven needs a compiler, and asserting the JDK is what catches a
# JAVA_HOME pointed at a JRE - which is exactly what the old hardcoded value was.
assert os.path.isfile(os.path.join(JAVA_HOME, 'bin', 'javac')), \
    f'no bin/javac under {JAVA_HOME} - is this a JRE?'
print('JAVA_HOME =', JAVA_HOME)
!{JAVA_HOME}/bin/java -version 2>&1
!{JAVA_HOME}/bin/javac -version 2>&1

env: JAVA_HOME=/usr/lib/jvm/jre-11-openjdk


Set the Maven environment and the `PATH` used for the build.

`MAVEN_HOME` is read from the environment, and `PATH` is rebuilt from `JAVA_HOME` and
`MAVEN_HOME` plus the system directories. Anaconda is deliberately left out so the
build uses the same tooling the nightly cron job does.

In [4]:
# MAVEN_HOME comes from the environment too, e.g.
#   export MAVEN_HOME=/opt/maven
MAVEN_HOME = os.environ.get('MAVEN_HOME', '').strip()
assert MAVEN_HOME, 'MAVEN_HOME is not set - export it before launching jupyter'
assert os.path.isfile(os.path.join(MAVEN_HOME, 'bin', 'mvn')), \
    f'no bin/mvn under {MAVEN_HOME}'

# Anaconda is deliberately absent, matching the PATH the nightly cron job uses.
os.environ['PATH'] = os.pathsep.join([
    os.path.join(JAVA_HOME, 'bin'),
    os.path.join(MAVEN_HOME, 'bin'),
    '/usr/local/sbin', '/usr/local/bin', '/usr/sbin', '/usr/bin',
])
print('PATH =', os.environ['PATH'])
!which java mvn git
!mvn -version

env: MAVEN_HOME=/opt/maven
env: PATH=/opt/apache-maven-3.6.0/bin/:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin


Prepare the target directory and clone the cytoscape git repo.

In [5]:
print('Changing to directory: ' + NOTEBOOK_DIR)
os.chdir(NOTEBOOK_DIR)

# Point to build location (the directory to clone parent cytoscape into)
BUILD_PARENT_DIR = os.path.join(os.getcwd(), 'release-build')
if not os.path.exists(BUILD_PARENT_DIR):
    os.mkdir(BUILD_PARENT_DIR)
else:
    shutil.rmtree(BUILD_PARENT_DIR)
    os.mkdir(BUILD_PARENT_DIR)

os.chdir(BUILD_PARENT_DIR)
![[ -d cytoscape ]] || git clone https://github.com/cytoscape/cytoscape
CYTOSCAPE_ROOT_DIR = os.path.join(BUILD_PARENT_DIR, 'cytoscape')
# cy.sh clones the sub projects into the top level clone itself, so the build
# directory and the top level clone are the same directory.
CYTOSCAPE_DIR = CYTOSCAPE_ROOT_DIR

def cd(directory=BUILD_PARENT_DIR, *subdirs):
    if subdirs:
        directory = os.path.join(directory, *subdirs)
    if os.getcwd() != directory:
        os.chdir(directory)

Changing to directory: /home/cybuilder/cytoscape-admin-scripts
Cloning into 'cytoscape'...
remote: Enumerating objects: 577, done.
remote: Total 577 (delta 0), reused 0 (delta 0), pack-reused 577
Receiving objects: 100% (577/577), 758.90 KiB | 0 bytes/s, done.
Resolving deltas: 100% (284/284), done.


In [6]:
# develop for a major release, release/3.X.X for a minor one.
#
# cy.sh is versioned alongside the code it builds, so this also selects which cy.sh
# runs.  It must name a branch whose cy.sh includes
#   https://github.com/cytoscape/cytoscape/pull/34
# which is commit a9ce955f0cbd09c984bb843a7560850a1a7e4d05 on develop.  Older
# branches carry a cy.sh with no 'pull' that clones and no 'build' command.  The
# next section checks this; to check a branch beforehand, run:
#   git merge-base --is-ancestor a9ce955f0cbd09c984bb843a7560850a1a7e4d05 origin/<branch>
STARTING_BRANCH = os.environ.get('STARTING_BRANCH', '').strip()
assert STARTING_BRANCH, 'STARTING_BRANCH is not set - export it before launching jupyter'
print('STARTING_BRANCH =', STARTING_BRANCH)

## 2. Pull the develop branch of Cytoscape

Note that to execute the checkout step here, you will need to have set up an SSH key on this machine that does not use password validation, or else parts of the build will be stall when they require input. 

In [ ]:
cd(CYTOSCAPE_ROOT_DIR)
!git checkout {STARTING_BRANCH}

## 2a. Verify this branch has the required cy.sh

In [ ]:
# Stop here rather than part way through the build.  On a branch whose cy.sh predates
# this commit, 'pull' pulls the repositories that already exist but clones none, and
# 'build' then reports "Invalid command build" - neither of which says "wrong branch".
CY_SH_COMMIT = 'a9ce955f0cbd09c984bb843a7560850a1a7e4d05'

rc = Popen(['git', 'merge-base', '--is-ancestor', CY_SH_COMMIT, 'HEAD'],
           cwd=CYTOSCAPE_ROOT_DIR).wait()
assert rc == 0, (
    f'{STARTING_BRANCH} does not contain the cy.sh from '
    'https://github.com/cytoscape/cytoscape/pull/34 - it has no \'pull\' that clones '
    'and no \'build\' command, so this notebook cannot build from it.')

## 2b. Clone and update the sub projects

In [ ]:
# 'pull' clones any sub project that is not here yet and updates the rest.  It puts new
# clones on develop whatever STARTING_BRANCH is, so 'switch' is what moves them onto
# the target branch.
cd(CYTOSCAPE_ROOT_DIR)
!./cy.sh pull
!./cy.sh switch {STARTING_BRANCH}

## 2c. Confirm every repository is on the target branch

In [ ]:
# cy.sh reports a failed checkout but still exits 0, so confirm the result rather than
# trusting it: a repository left on another branch would be built and published as part
# of this release.  This list is the set of repositories cy.sh manages, so a sub project
# that never got cloned is caught here too.
cd(CYTOSCAPE_DIR)
REPOSITORIES = ['.', 'parent', 'api', 'impl', 'support', 'gui-distribution', 'app-developer']

mismatched = {}
for repo in REPOSITORIES:
    branch = Popen(['git', 'symbolic-ref', '--short', '-q', 'HEAD'],
                   stdout=PIPE,
                   cwd=os.path.join(CYTOSCAPE_DIR, repo)).communicate()[0].decode().strip()
    print(f'{repo:<20} {branch or "(detached HEAD)"}')
    if branch != STARTING_BRANCH:
        mismatched[repo] = branch or 'detached HEAD'

assert not mismatched, f'not on {STARTING_BRANCH}: {mismatched}'

## 2d. Reset

In [8]:
cd(CYTOSCAPE_DIR)
!./cy.sh run-all 'git clean -f -d'
!./cy.sh run-all 'git reset --hard'

------------------------------------------------------------------------
Executing command: git clean -f -d
--in .
------------------------------------------------------------------------
--in parent
------------------------------------------------------------------------
--in api
------------------------------------------------------------------------
--in impl
------------------------------------------------------------------------
--in support
------------------------------------------------------------------------
--in gui-distribution
------------------------------------------------------------------------
--in app-developer
------------------------------------------------------------------------
------------------------------------------------------------------------
Executing command: git reset --hard
--in .
HEAD is now at 5f266cb Update README.md
------------------------------------------------------------------------
--in parent
HEAD is now at 8a407bd Update mockito version
--

## 2e. Verify status

In [9]:
cd(CYTOSCAPE_DIR)
!./cy.sh run-all 'git status'

------------------------------------------------------------------------
Executing command: git status
--in .
# On branch develop
nothing to commit, working directory clean
------------------------------------------------------------------------
--in parent
# On branch develop
nothing to commit, working directory clean
------------------------------------------------------------------------
--in api
# On branch develop
nothing to commit, working directory clean
------------------------------------------------------------------------
--in impl
# On branch develop
nothing to commit, working directory clean
------------------------------------------------------------------------
--in support
# On branch develop
nothing to commit, working directory clean
------------------------------------------------------------------------
--in gui-distribution
# On branch develop
nothing to commit, working directory clean
------------------------------------------------------------------------
--in app

## 3. Build Cytoscape and ensure no errors

This may take a while. Expect to build subrepos first before building from the root directory

Currently, you should expect two or so non-fatal javadoc-bundle-options related errors.

In [10]:
# 'build' rather than a plain 'mvn install': api and impl fork a generate-sources
# lifecycle that resolves from the local maven repository instead of the reactor, so
# the inner projects have to be built first for a cold repository to work at all.
# stderr is merged into the log because cy.sh reports its own failures there.
cd(CYTOSCAPE_DIR)
with open('build_output.txt', 'w') as outf:
    process = Popen(['./cy.sh', 'build'],
                stdout=outf,
                stderr=STDOUT,
                cwd=CYTOSCAPE_DIR)
    rc = process.wait()

print("Showing ERROR lines in build...")
!cat build_output.txt | grep ERROR

assert rc == 0, f'./cy.sh build failed with exit code {rc} - see build_output.txt'

Showing ERROR lines in build...
[ERROR] Error fetching link: /home/cybuilder/cytoscape-admin-scripts/release-build/cytoscape/cytoscape/api/app-api/target/javadoc-bundle-options. Ignored it.
[ERROR] Error fetching link: /home/cybuilder/cytoscape-admin-scripts/release-build/cytoscape/cytoscape/api/swing-app-api/target/javadoc-bundle-options. Ignored it.


## 4. Build Cytoscape installers

This requires Install4J to be configured on your machine and for the Install4j project file to point to the correct Mac signing file. The keystore password is set below (blank in this config), to be interpreted by the Install4j Maven plugin.

In [11]:
cd(CYTOSCAPE_DIR, 'gui-distribution', 'packaging')
%env MAC_KEYSTORE_PASSWORD=
!mvn clean install -U

env: MAC_KEYSTORE_PASSWORD=
[INFO] Scanning for projects...
[WARNING] 
[WARNING] Some problems were encountered while building the effective model for org.cytoscape.distribution:packaging:jar:3.9.0-SNAPSHOT
[WARNING] 'build.plugins.plugin.version' for org.sonatype.install4j:install4j-maven-plugin is missing. @ org.cytoscape.distribution:packaging:[unknown-version], /home/cybuilder/cytoscape-admin-scripts/release-build/cytoscape/cytoscape/gui-distribution/packaging/pom.xml, line 104, column 15
[WARNING] 
[WARNING] It is highly recommended to fix these problems because they threaten the stability of your build.
[WARNING] 
[WARNING] For this reason, future Maven versions might no longer support building such malformed projects.
[WARNING] 
[INFO] 
[INFO] ----------------< org.cytoscape.distribution:packaging >----------------
[INFO] Building Cytoscape Release Packaging 3.9.0-SNAPSHOT
[INFO] --------------------------------[ jar ]---------------------------------
Downloaded from central: ht

[INFO] No tests to run.
[INFO] 
[INFO] --- jacoco-maven-plugin:0.8.4:report (report) @ packaging ---
[INFO] Skipping JaCoCo execution due to missing execution data file.
[INFO] 
[INFO] --- maven-jar-plugin:2.4:jar (default-jar) @ packaging ---
[INFO] Building jar: /home/cybuilder/cytoscape-admin-scripts/release-build/cytoscape/cytoscape/gui-distribution/packaging/target/packaging-3.9.0-SNAPSHOT.jar
[INFO] 
[INFO] >>> maven-source-plugin:3.0.1:jar (attach-sources) > generate-sources @ packaging >>>
[INFO] 
[INFO] --- jacoco-maven-plugin:0.8.4:prepare-agent (default) @ packaging ---
[INFO] argLine set to -javaagent:/home/cybuilder/.m2/repository/org/jacoco/org.jacoco.agent/0.8.4/org.jacoco.agent-0.8.4-runtime.jar=destfile=/home/cybuilder/cytoscape-admin-scripts/release-build/cytoscape/cytoscape/gui-distribution/packaging/target/jacoco.exec -Djdk.net.URLClassPath.disableClassPathURLCheck=true
[INFO] 
[INFO] <<< maven-source-plugin:3.0.1:jar (attach-sources) < generate-sources @ packaging 

[INFO] 
[INFO] --- maven-install-plugin:2.4:install (default-install) @ packaging ---
[INFO] Installing /home/cybuilder/cytoscape-admin-scripts/release-build/cytoscape/cytoscape/gui-distribution/packaging/target/packaging-3.9.0-SNAPSHOT.jar to /home/cybuilder/.m2/repository/org/cytoscape/distribution/packaging/3.9.0-SNAPSHOT/packaging-3.9.0-SNAPSHOT.jar
[INFO] Installing /home/cybuilder/cytoscape-admin-scripts/release-build/cytoscape/cytoscape/gui-distribution/packaging/pom.xml to /home/cybuilder/.m2/repository/org/cytoscape/distribution/packaging/3.9.0-SNAPSHOT/packaging-3.9.0-SNAPSHOT.pom
[INFO] Installing /home/cybuilder/cytoscape-admin-scripts/release-build/cytoscape/cytoscape/gui-distribution/packaging/target/packaging-3.9.0-SNAPSHOT-sources.jar to /home/cybuilder/.m2/repository/org/cytoscape/distribution/packaging/3.9.0-SNAPSHOT/packaging-3.9.0-SNAPSHOT-sources.jar
[INFO] ------------------------------------------------------------------------
[INFO] BUILD SUCCESS
[INFO] --------

## 5. Copying Cytoscape installers to weekly download page

When completed installer executables can be found in `cytoscape/gui-distribution/packaging/target/media`
and compressed builds for Linux and Windows can be found in `cytoscape/gui-distribution/assembly/target`

all of which are copied to `$PUBLISH_ROOT/Cytoscape-<version>/<DATE>`, where the version
comes from the `pom.xml` of the tree that was just built.

In [ ]:
# Publish under Cytoscape-<version>, taking the version from the tree that was just
# built rather than a hardcoded number.  Printed here so the destination is visible
# before the next cell writes to the web root.
NS = {'m': 'http://maven.apache.org/POM/4.0.0'}
_pom = ET.parse(os.path.join(CYTOSCAPE_DIR, 'pom.xml')).getroot()
_ver = (_pom.findtext('m:version', namespaces=NS)
        or _pom.findtext('m:parent/m:version', namespaces=NS))
assert _ver, f'no version found in {CYTOSCAPE_DIR}/pom.xml'

RELEASE_VERSION = _ver.strip().replace('-SNAPSHOT', '')      # 3.11.0-SNAPSHOT -> 3.11.0
PUBLISH_ROOT = os.environ.get('PUBLISH_ROOT', '/var/www/html/cytoscape-builds')
DATED_DIR = os.path.join(PUBLISH_ROOT, f'Cytoscape-{RELEASE_VERSION}',
                         f'{datetime.date.today():%Y_%m_%d}')
print('publish to:', DATED_DIR)

In [12]:
!mkdir -p {DATED_DIR}
!rsync -av --progress {CYTOSCAPE_DIR}/gui-distribution/packaging/target/media/* {DATED_DIR}
!rsync -av --progress {CYTOSCAPE_DIR}/gui-distribution/assembly/target/*.{{gz,zip}} {DATED_DIR}

sending incremental file list
Cytoscape_3_9_0-SNAPSHOT_macos.dmg
    292,894,009 100%  208.12MB/s    0:00:01 (xfr#1, to-chk=7/8)
Cytoscape_3_9_0-SNAPSHOT_unix.sh
    290,773,961 100%  163.02MB/s    0:00:01 (xfr#2, to-chk=6/8)
Cytoscape_3_9_0-SNAPSHOT_windows_32bit.exe
    292,420,096 100%  143.38MB/s    0:00:01 (xfr#3, to-chk=5/8)
Cytoscape_3_9_0-SNAPSHOT_windows_64bit.exe
    292,527,104 100%  131.04MB/s    0:00:02 (xfr#4, to-chk=4/8)
md5sums
            290 100%    2.18kB/s    0:00:00 (xfr#5, to-chk=3/8)
output.txt
            813 100%    6.11kB/s    0:00:00 (xfr#6, to-chk=2/8)
sha256sums
            418 100%    3.14kB/s    0:00:00 (xfr#7, to-chk=1/8)
updates.xml
          1,738 100%   13.06kB/s    0:00:00 (xfr#8, to-chk=0/8)

sent 1,168,904,326 bytes  received 168 bytes  212,528,089.82 bytes/sec
total size is 1,168,618,429  speedup is 1.00
sending incremental file list
cytoscape-unix-3.9.0-SNAPSHOT.tar.gz
    287,992,446 100%  238.80MB/s    0:00:01 (xfr#1, to-chk=1/2)
cytoscape-wind

## 6. Notarizing on idekerlab-macmini

The install4j generated .dmg must be notarized to run on macOS 10.15 and above. The .dmg will be sent to idekerlab-macmini and submitted for notarization. An email will be sent to William Markuske when the notarization process is complete. As soon as the .dmg is notarized it will work and no further action is needed.

In [13]:
!rsync -av --progress {DATED_DIR}/*.dmg idekerlab@idekerlab-macmini.ucsd.edu:~/apps_to_notarize/
!ssh idekerlab@idekerlab-macmini.ucsd.edu '~/notarizedmg_jing.sh ~/apps_to_notarize/*.dmg Snapshot.$(date +%Y%m%d)'    

building file list ... 
1 file to consider
Cytoscape_3_9_0-SNAPSHOT_macos.dmg
    292,894,009 100%   84.61MB/s    0:00:03 (xfr#1, to-chk=0/1)

sent 292,929,916 bytes  received 46 bytes  65,095,547.11 bytes/sec
total size is 292,894,009  speedup is 1.00
No errors uploading '/Users/idekerlab/apps_to_notarize/Cytoscape_3_9_0-SNAPSHOT_macos.dmg'.
RequestUUID = 6a41ff2a-9fd8-43d6-b378-adfd075b613b




You can check the status of the notarization by pasting in the `RequestUUID` value into the following command and running.

In [16]:
!ssh idekerlab@idekerlab-macmini.ucsd.edu '~/notarizestatus_jing.sh 6a41ff2a-9fd8-43d6-b378-adfd075b613b'

No errors getting notarization info.

          Date: 2021-04-30 13:56:52 +0000
          Hash: 7d32053845cd00b39f1a454710f3a650974f7f5d980ea41753a515efb27ee8cc
    LogFileURL: https://osxapps-ssl.itunes.apple.com/itunes-assets/Enigma115/v4/91/1f/c3/911fc3d0-a26b-067b-633c-b9cc8ce3e71a/developer_log.json?accessKey=1619985924_8906453629669302967_JpRk3DbwQdUPeY0fjgXAfuB0rVc%2FQkGm0wHltInIzu7k9XgkuwZmK3DtsDmyD%2FHD%2FAP1BLjqG%2BlEIEvOJNlbOcYPq%2FKsH%2B5LfUd%2Byja0avqDGmZ15ZgRo4Rp9sOJYsf%2Fv8PydAzkQSHl9NQDC8CfSUg%2BE1%2Fpw47CEbwJpnrmmfQ%3D
   RequestUUID: 6a41ff2a-9fd8-43d6-b378-adfd075b613b
        Status: success
   Status Code: 0
Status Message: Package Approved

